<a href="https://colab.research.google.com/github/royalsflush/hackernews_vs_googletrends/blob/main/Hacker_news.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Setup, run me first
from google.cloud import bigquery
import pandas as pd

from google.colab import auth
auth.authenticate_user()
print('Authenticated')

project_id = 'data-mining-project-505611'
project_number = 755541129993
client = bigquery.Client(project=project_id)

import google.api_core.exceptions
from google.cloud import bigquery_connection_v1

client_connection = bigquery_connection_v1.ConnectionServiceClient()

def create_connection(
    project_id: str,
    location: str,
    connection_id: str,
):
    """Creates a BigQuery connection to a Cloud Resource.

    Cloud Resource connection creates a service account which can then be
    granted access to other Google Cloud resources for federated queries.

    Args:
        project_id: The Google Cloud project ID.
        location: The location of the connection (for example, "us-central1").
        connection_id: The ID of the connection to create.
    """

    parent = client_connection.common_location_path(project_id, location)

    connection = bigquery_connection_v1.Connection(
        friendly_name="Example Connection",
        description="A sample connection for a Cloud Resource.",
        cloud_resource=bigquery_connection_v1.CloudResourceProperties(),
    )

    try:
        created_connection = client_connection.create_connection(
            parent=parent, connection_id=connection_id, connection=connection
        )
        print(f"Successfully created connection: {created_connection.name}")
        print(f"Friendly name: {created_connection.friendly_name}")
        print(
            f"Service Account: {created_connection.cloud_resource.service_account_id}"
        )

    except google.api_core.exceptions.AlreadyExists:
        print(f"Connection with ID '{connection_id}' already exists.")
        print("Please use a different connection ID.")
    except Exception as e:
        print(f"An unexpected error occurred while creating the connection: {e}")

create_connection(
    project_id=project_id,
    location="US",
    connection_id="hackernews_royalsflush",
)

Authenticated
Connection with ID 'hackernews_royalsflush' already exists.
Please use a different connection ID.


In [6]:
!gcloud projects add-iam-policy-binding '755541129993' --member='serviceAccount:bqcx-755541129993-n51b@gcp-sa-bigquery-condel.iam.gserviceaccount.com' --role='roles/aiplatform.user' --condition=None

Updated IAM policy for project [755541129993].
bindings:
- members:
  - serviceAccount:bqcx-755541129993-n51b@gcp-sa-bigquery-condel.iam.gserviceaccount.com
  role: roles/aiplatform.user
- members:
  - serviceAccount:service-755541129993@gcp-sa-csc-hpsa.iam.gserviceaccount.com
  role: roles/cloudsecuritycompliance.serviceAgent
- members:
  - serviceAccount:service-755541129993@containerregistry.iam.gserviceaccount.com
  role: roles/containerregistry.ServiceAgent
- members:
  - serviceAccount:service-755541129993@gcp-sa-dataform.iam.gserviceaccount.com
  role: roles/dataform.serviceAgent
- members:
  - user:thomas@royalsflush.com
  role: roles/owner
- members:
  - serviceAccount:service-755541129993@gcp-sa-pubsub.iam.gserviceaccount.com
  role: roles/pubsub.serviceAgent
- members:
  - serviceAccount:service-755541129993@serverless-robot-prod.iam.gserviceaccount.com
  role: roles/run.serviceAgent
etag: BwZZU6ZQtGU=
version: 1


In [5]:
# Create story view
query = """
CREATE OR REPLACE VIEW hackernews_royalsflush.hackernews_story AS (
  SELECT title,
        url,
        text,
        `by`,
        score,
        `timestamp`,
        id,
        descendants
  FROM `bigquery-public-data.hacker_news.full`
  WHERE type = 'story'
  AND `by` IS NOT NULL
  AND dead IS NULL
);
"""

resp = client.query(query)
print(resp)

QueryJob<project=data-mining-project-505611, location=US, id=d8ab6232-5bbb-4f20-bc2a-c90a9d76a697>


In [6]:
# Create comments view
query = """
CREATE OR REPLACE VIEW hackernews_royalsflush.hackernews_comment AS (
  SELECT text,
       `by`,
       `timestamp`,
       id,
       parent,
  FROM `bigquery-public-data.hacker_news.full`
  WHERE type = 'comment'
  AND `by` IS NOT NULL
  AND dead IS NULL
);
"""

resp = client.query(query)
print(resp)

QueryJob<project=data-mining-project-505611, location=US, id=89c7f1ce-9bd7-4c2a-a77b-4b9604ae47c0>


In [8]:
# Create story <-> comment ID mapping
query = """
CREATE OR REPLACE TABLE hackernews_royalsflush.hackernews_comment_story_mapping AS (
  WITH RECURSIVE
  comment_id_pairs AS (
    SELECT id, parent
    FROM `data-mining-project-505611.hackernews_royalsflush.hackernews_comment`
  ),
  story_ids AS (
    SELECT id
    FROM `data-mining-project-505611.hackernews_royalsflush.hackernews_story`
  ),
  R AS (
    (SELECT id, parent AS ancestor from comment_id_pairs)
    UNION ALL (
      SELECT R.id, comment_id_pairs.parent AS ancestor
      FROM R
      INNER JOIN comment_id_pairs ON ancestor = comment_id_pairs.id
    )
  )
  SELECT R.id, ancestor
  FROM R
  INNER JOIN story_ids
  ON story_ids.id = R.ancestor
);
"""

resp = client.query(query)
print(resp)

QueryJob<project=data-mining-project-505611, location=US, id=b4eb818a-a4cf-4547-b118-257437e6d978>


In [7]:
# Calculates the popularity for a particular story bucketed by month.
query = """
CREATE OR REPLACE TABLE hackernews_royalsflush.hackernews_popularity AS (
  SELECT story_id,
        month,
        SUM(popularity) AS popularity
  FROM (
    SELECT story_id,
          DATETIME_TRUNC(story_timestamp, MONTH) AS month,
          story_score AS popularity,
          "story" AS type
    FROM `hackernews_royalsflush.hackernews_payload` AS story_payload
    UNION ALL
    SELECT story_id,
          DATETIME_TRUNC(comment_timestamp, MONTH) as month,
          1 AS popularity,
          "comment" AS type
    FROM `hackernews_royalsflush.hackernews_payload` AS story_payload,
      UNNEST(comment_timestamps) AS comment_timestamp
  )
  GROUP BY story_id, month
);
"""

resp = client.query(query)
print(resp)

QueryJob<project=data-mining-project-505611, location=US, id=2f0293ac-51b1-4eb6-8381-a88cb2d4093a>


In [11]:
# Queries from now on take the absolute part of forever, so I need to sample things.

# Tried using https://docs.cloud.google.com/bigquery/docs/table-sampling, didn't
# quite work, didn't return anything I thought it should
query = """
CREATE OR REPLACE TABLE hackernews_royalsflush.hackernews_sampled_stories AS (
  SELECT id AS story_id
  FROM `hackernews_royalsflush.hackernews_story`
  WHERE rand() < 0.001
);
"""

resp = client.query(query)
print(resp)

QueryJob<project=data-mining-project-505611, location=US, id=4f3432bf-1a08-4492-9611-2b29d538fc5a>


In [12]:
# This is slow
import concurrent.futures
import requests
import threading
import time
from tqdm import tqdm

query = """
SELECT id, url
FROM `hackernews_royalsflush.hackernews_story`
WHERE url IS NOT NULL
AND id IN (SELECT story_id FROM hackernews_royalsflush.hackernews_sampled_stories)
"""

df = client.query(query).to_dataframe()
df.set_index('id', inplace=True)
df_lock = threading.Lock()
print(df)

headers = {
    'User-Agent': 'My User Agent 1.0',
    'From': 'youremail@domain.example'  # This is another valid field
}

# Modified example from ThreadPoolExecutor docs
def load_url(url, timeout):
    return requests.get(url, timeout=timeout, headers=headers)

start = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=100) as executor:
    # Start the load operations and mark each future with its URL
    future_to_url_id = {executor.submit(load_url, row['url'], 60): id for id, row in df.iterrows()}
    for future in tqdm(concurrent.futures.as_completed(future_to_url_id), total=len(df)):
        id  = future_to_url_id[future]

        try:
            data = future.result()
        except Exception as exc:
            with df_lock:
                df.loc[id, 'content'] = ""
                df.loc[id, 'url_error'] = True
        else:
            with df_lock:
                df.loc[id, 'content'] = data.text
                df.loc[id, 'url_error'] = False
print(f"URL fetching time: {time.time() - start}")

start = time.time()
load_job = client.load_table_from_dataframe(df, 'hackernews_royalsflush.hackernews_story_content')
print(f"BigQuery write time: {time.time() - start}")
print(resp)

                                                        url
id                                                         
37894411  https://techcrunch.com/2023/10/11/instagram-he...
37899107  https://www.bleepingcomputer.com/news/microsof...
37942290  https://www.glamour.com/story/welcome-to-the-s...
37811204  https://www.reuters.com/technology/hackers-adv...
37821275  https://kentherecruiter.substack.com/p/why-law...
...                                                     ...
13037389             https://github.com/sakurity/truefactor
1300635   http://www.stevesouders.com/blog/2010/04/26/ca...
13063194  http://www.nytimes.com/2016/11/29/world/americ...
12991292  http://www.lansingstatejournal.com/story/news/...
13097540  http://blogs.exeter.ac.uk/stoicismtoday/2016/1...

[4311 rows x 1 columns]


100%|██████████| 4311/4311 [13:09<00:00,  5.46it/s]


URL fetching time: 793.6918308734894
BigQuery write time: 24.24409556388855
QueryJob<project=data-mining-project-505611, location=US, id=4f3432bf-1a08-4492-9611-2b29d538fc5a>


In [16]:
# Aggregates all the information needed to calculate the popularity score
# and the topics
query = """
CREATE OR REPLACE TABLE hackernews_royalsflush.hackernews_full AS (
  SELECT story.id AS story_id,
        CONCAT("https://news.ycombinator.com/item?id=", id) AS story_link,
        score AS story_score,
        story.timestamp AS story_timestamp,
        title AS story_title,
        IFNULL(text, "") AS story_text,
        "" AS url_content,
        IFNULL(ARRAY_TO_STRING(comment.comment_text, "\\n", ""), "") AS comments_text,
        comment.comment_ids,
        comment.comment_timestamps
  FROM `hackernews_royalsflush.hackernews_story` AS story
  LEFT JOIN (
        SELECT ARRAY_AGG(comment.id) AS comment_ids,
              ARRAY_AGG(text IGNORE NULLS) AS comment_text,
              ARRAY_AGG(`timestamp` IGNORE NULLS) AS comment_timestamps,
              mapping.ancestor AS story_id
        FROM `hackernews_royalsflush.hackernews_comment` AS comment
        JOIN `hackernews_royalsflush.hackernews_comment_story_mapping` AS mapping
        ON mapping.id = comment.id
        GROUP BY mapping.ancestor
  ) AS comment
  ON comment.story_id = story.id
);
"""

resp = client.query(query)
print(resp)

QueryJob<project=data-mining-project-505611, location=US, id=2d23e0ef-06d4-4016-816a-d7ca223ca202>


In [9]:
import tensorflow as tf
import tensorflow_hub as hub
import time

embed = hub.load("https://tfhub.dev/google/universal-sentence-encoder/4")

query = """
SELECT story_id,
       story_title,
       story_text,
       url_content,
       IFNULL(comments_text, "") AS comments_text,
FROM `hackernews_royalsflush.hackernews_full`
WHERE story_id IN (SELECT story_id FROM hackernews_royalsflush.hackernews_sampled_stories);
"""

df = client.query(query).to_dataframe()
df.set_index('story_id', inplace=True)
print(df)

def get_embed_title(story_title):
  return embed(story_title)

def get_embed_text(text, delimiter="\n"):
    @tf.function
    def _map_fn(a):
        t = tf.cast(a, tf.string)
        t = tf.strings.split(t, sep=delimiter)
        e = embed(t)
        # Changes this multidimensional vector into its mean
        # representation
        e = tf.reduce_mean(e, axis=0)
        # Removes dimensions of size 1 from the tensor
        return tf.squeeze(e)
    return tf.map_fn(_map_fn, text, dtype=tf.float32, parallel_iterations=100)


def process(metadata):
    start = time.time()
    title_embed = get_embed_title(metadata['story_title'])
    print(f"Title embeddings processing time: {time.time() - start}")

    start = time.time()
    comment_embed = get_embed_text(metadata['comments_text'])
    print(f"Comment embeddings processing time: {time.time() - start}")

    start = time.time()
    story_text_embed = get_embed_text(metadata['story_text'])
    print(f"Story text embeddings processing time: {time.time() - start}")

    return {
        'title_embed': title_embed,
        'story_text_embed': story_text_embed,
        'comment_embed': comment_embed
    }

start = time.time()
r = process(df)
print(f"Embeddings processing time: {time.time() - start}")

embeddings = pd.DataFrame(index=df.index)

# Dimensions are 512, always
for i in range(512):
  for feature in ['title_embed', 'story_text_embed', 'comment_embed']:
    embeddings[f"{feature}_{i}"] = r[f"{feature}"].numpy()[:, i]
print(embeddings)

start = time.time()
load_job = client.load_table_from_dataframe(embeddings, 'hackernews_royalsflush.hackernews_embeddings')
print(f"BigQuery write time: {time.time() - start}")

                                                story_title  \
story_id                                                      
34380358                           Washing Machine Settings   
38515351                              Starring the Computer   
1075996                India: The New Land of Opportunity?    
27310255                 Apply for Our UI Design Internship   
21415206  Browser Extension That Funds Environmental Pro...   
...                                                     ...   
11623633  High Performance ClojureScript with WebGL, Asm...   
16633477  These early humans survived a supervolcano eru...   
25688970  Racial Justice Requires Ending Drug War, Say L...   
28080511  LinkedIn Competitor Polywork Raises $13MM from...   
30198620  Amazon is ramping up its push for legalizing m...   

                                                 story_text url_content  \
story_id                                                                  
34380358                      

/tmp/ipykernel_1156/1430860946.py:66: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  embeddings[f"{feature}_{i}"] = r[f"{feature}"].numpy()[:, i]
/tmp/ipykernel_1156/1430860946.py:66: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  embeddings[f"{feature}_{i}"] = r[f"{feature}"].numpy()[:, i]
/tmp/ipykernel_1156/1430860946.py:66: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1

          title_embed_0  story_text_embed_0  comment_embed_0  title_embed_1  \
story_id                                                                      
34380358      -0.012363           -0.035657        -0.002380       0.045893   
38515351      -0.019183           -0.035657         0.004637       0.012537   
1075996       -0.042567           -0.035657        -0.035657      -0.077497   
27310255      -0.005656           -0.032512        -0.035657      -0.030298   
21415206       0.055774           -0.035657        -0.035657       0.068644   
...                 ...                 ...              ...            ...   
11623633      -0.009860           -0.035657        -0.035657       0.028204   
16633477       0.021761           -0.035657        -0.035657      -0.042647   
25688970      -0.009480           -0.035657        -0.035657      -0.083619   
28080511       0.029510           -0.035657        -0.035657      -0.052527   
30198620       0.014710           -0.035657        -

In [10]:
# Train the model

query = """
CREATE OR REPLACE MODEL `hackernews_royalsflush.clustering`
OPTIONS (
  MODEL_TYPE='KMEANS',
  num_trials=10,
  max_parallel_trials=2,
  HPARAM_TUNING_OBJECTIVES=['DAVIES_BOULDIN_INDEX'])
AS (
  SELECT *
  EXCEPT (story_id)
  FROM hackernews_royalsflush.hackernews_embeddings
);
"""

resp = client.query(query)
print(resp)

QueryJob<project=data-mining-project-505611, location=US, id=1304daf4-a373-4727-9d1b-3d88024048f1>


In [ ]:
query = """
SELECT story_id,
       CENTROID_ID,
       NEAREST_CENTROIDS_DISTANCE
FROM ML.PREDICT(MODEL `data-mining-project-505611.hackernews_royalsflush.clustering`, (
  SELECT *
  FROM `hackernews_royalsflush.hackernews_embeddings`
))
"""

resp = client.query(query)
print(resp)

In [ ]:
!pip install pytrends-modern

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 761.1 kB/s eta 0:00:00


In [ ]:
from pytrends_modern import TrendReq

pytrends = TrendReq(hl='en-US', tz=0)
pytrends.build_payload(
    kw_list=['Python', 'JavaScript'],
    timeframe='all',
    geo='', # Default is world
)

interest_df = pytrends.interest_over_time()
related = pytrends.related_queries()

print(interest_df)
print(related)

            Python  JavaScript  isPartial
date                                     
2004-01-01      24          93      False
2004-02-01      23          94      False
2004-03-01      23         100      False
2004-04-01      24          98      False
2004-05-01      22          87      False
...            ...         ...        ...
2026-04-01      75          14      False
2026-05-01      84          17      False
2026-06-01      88          19      False
2026-07-01      65          14      False
2026-08-01      48           8       True

[272 rows x 3 columns]
{'Python': {'top':                  query  value
0           python for    100
1          python list     75
2          python code     43
3       install python     34
4            python if     34
5         monty python     30
6        online python     30
7       what is python     29
8      download python     28
9       python windows     24
10              pandas     23
11        python array     23
12       pandas pytho

# Appendix

Attempts that didn't work

In [ ]:
# Creates the embeddings and runs the clustering
#
# Rejected, this for some reason takes too long, even when limited by 10k
# samples. My humble TF embeddings calculation fares better.

query = """
CREATE OR REPLACE MODEL
`data-mining-project-505611.hackernews_royalsflush.clustering`
REMOTE WITH CONNECTION `projects/data-mining-project-505611/locations/us/connections/hackernews_royalsflush`
OPTIONS (ENDPOINT = 'gemini-embedding-001');
"""

resp = client.query(query)
print(resp)

query = """
SELECT *
FROM
  AI.GENERATE_EMBEDDING(
    MODEL hackernews_royalsflush.clustering,
    (
      SELECT story_title AS title,
              CONCAT(story_text, url_content, comments_text) AS content
      FROM hackernews_royalsflush.hackernews_full
    ),
      STRUCT('CLUSTERING' AS task_type)
);
"""